# Mesoscale and topology with MuxVizPy

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import graph_tool as gt
import graph_tool.draw as gtdraw

from MuxVizPy.utils import parsing
from MuxVizPy import topology, mesoscale, versatility

np.random.seed(42)
gt.seed_rng(42)
plt.rcParams["figure.dpi"] = 110

## Running example

The office network has 10 people in email, chat, and meetings. `supra` includes
categorical coupling. `supra_ec` contains only the observed layer edges.

In [ ]:
NODE_NAMES = ["Ada", "Ben", "Cleo", "Dan", "Eve", "Femi", "Gil", "Hana", "Ivan", "Jo"]
LAYER_NAMES = ["email", "chat", "meetings"]
N, L = len(NODE_NAMES), len(LAYER_NAMES)
idx = {name: i for i, name in enumerate(NODE_NAMES)}

INTRA = {
    "email": [
        ("Ada", "Ben"), ("Ada", "Cleo"), ("Ada", "Dan"), ("Ben", "Cleo"),
        ("Cleo", "Dan"), ("Dan", "Eve"), ("Eve", "Femi"), ("Femi", "Gil"),
        ("Gil", "Hana"), ("Hana", "Ivan"), ("Ivan", "Jo"), ("Jo", "Ada"),
    ],
    "chat": [
        ("Ada", "Ben"), ("Ben", "Eve"), ("Eve", "Hana"), ("Hana", "Jo"),
        ("Jo", "Cleo"), ("Cleo", "Femi"), ("Femi", "Ivan"), ("Ivan", "Dan"),
        ("Dan", "Gil"), ("Gil", "Ada"), ("Ben", "Hana"),
    ],
    "meetings": [
        ("Ada", "Ben"), ("Ada", "Cleo"), ("Ben", "Cleo"),
        ("Dan", "Eve"), ("Dan", "Femi"), ("Eve", "Femi"),
        ("Gil", "Hana"), ("Gil", "Ivan"), ("Hana", "Ivan"),
    ],
}

rows = []
for layer_name, pairs in INTRA.items():
    layer = LAYER_NAMES.index(layer_name)
    for a, b in pairs:
        rows.append((idx[a], layer, idx[b], layer, 1.0))
        rows.append((idx[b], layer, idx[a], layer, 1.0))   # undirected -> both directions

edges = pl.DataFrame(
    rows,
    schema=["node.from", "layer.from", "node.to", "layer.to", "weight"],
    orient="row",
)

t = parsing.build_tensor_from_dataframe(edges)
g_list = parsing.build_list_of_graphs_from_tensor(t)
node_tensor = parsing.get_node_tensor_from_network_list(g_list)

# Coupled categorical multiplex: layers linked, each person to their own replicas.
C_cat = parsing.build_interlayer_coupling_matrix(L, omega=1.0, kind="categorical")
supra = parsing.build_supra_adjacency_matrix_from_edge_colored_matrices(node_tensor, C_cat, N)

# Edge-coloured: the observed layers alone, with no coupling imposed.
supra_ec = parsing.build_supra_adjacency_matrix_from_tensor(t)

print(f"{N} nodes, {L} layers")
print(f"coupled        {supra.shape} nnz = {supra.nnz}")
print(f"edge-coloured  {supra_ec.shape} nnz = {supra_ec.nnz}"
      f"  ({supra.nnz - supra_ec.nnz} coupling edges fewer)")

## Connected components

`get_connected_components` finds connected parts in the coupled network.

In [ ]:
components = topology.get_connected_components(supra, nodes=N, layers=L)
n_components = len(np.unique(components))

print(f"number of components: {n_components}")
print("component per node:", components.astype(int))

In [ ]:
two_groups = {
    "day":   [(0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5)],
    "night": [(0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5)],
}
r = []
for li, (name, pairs) in enumerate(two_groups.items()):
    for a, b in pairs:
        r.append((a, li, b, li, 1.0))
        r.append((b, li, a, li, 1.0))
tg_edges = pl.DataFrame(r, schema=["node.from", "layer.from", "node.to", "layer.to", "weight"], orient="row")

tg_t = parsing.build_tensor_from_dataframe(tg_edges)
tg_nt = parsing.get_node_tensor_from_network_list(parsing.build_list_of_graphs_from_tensor(tg_t))
tg_C = parsing.build_interlayer_coupling_matrix(2, omega=1.0, kind="categorical")
tg_supra = parsing.build_supra_adjacency_matrix_from_edge_colored_matrices(tg_nt, tg_C, 6)

tg_components = topology.get_connected_components(tg_supra, nodes=6, layers=2)
print("two-triangle multiplex components:", tg_components.astype(int))
print(f"-> {len(np.unique(tg_components))} components, one per triangle")

### Three component definitions

`get_multi_LCC` uses the aggregate network. `get_multi_LIC` intersects each layer's
largest component. `get_multi_LVC` keeps pruning until the remaining nodes stay
connected inside every layer.

In [ ]:
lcc = topology.get_multi_LCC(g_list, obj_type="glist")
lic = topology.get_multi_LIC(g_list, obj_type="glist")
lvc = topology.get_multi_LVC(g_list, printt=False)   # printt=False silences its iteration log

for label, members in [("LCC", lcc), ("LIC", lic), ("LVC", lvc)]:
    print(f"{label}: {len(members):2d}/{N}  {sorted(NODE_NAMES[i] for i in members)}")

## Path statistics

`get_multi_path_statistics` returns physical-node distances, closeness, and one average
path length from the coupled matrix.

In [ ]:
stats = topology.get_multi_path_statistics(supra, nodes=N, layers=L)

closeness = np.array(stats["closeness"])
distance = stats["distance_matrix"]

print(f"average path length: {stats['avg_path_length']:.3f}")
print(f"diameter (longest shortest path): {int(distance.max())}")
print(f"most central person:  {NODE_NAMES[closeness.argmax()]} (closeness {closeness.max():.3f})")
print(f"least central person: {NODE_NAMES[closeness.argmin()]} (closeness {closeness.min():.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

im = axes[0].imshow(distance, cmap="viridis")
axes[0].set_xticks(range(N), NODE_NAMES, rotation=90)
axes[0].set_yticks(range(N), NODE_NAMES)
axes[0].set_title("Shortest-path distance")
for i in range(N):
    for j in range(N):
        axes[0].text(j, i, int(distance[i, j]), ha="center", va="center",
                     color="w" if distance[i, j] < distance.max() * 0.6 else "k", fontsize=7)
fig.colorbar(im, ax=axes[0], shrink=0.8)

order = np.argsort(closeness)
axes[1].barh(range(N), closeness[order])
axes[1].set_yticks(range(N), [NODE_NAMES[i] for i in order])
axes[1].set_xlabel("closeness")
axes[1].set_title("Who is central")
plt.tight_layout()
plt.show()

## Coreness across layers

`get_multi_Kcore_centrality` returns each node's smallest core index across the layers.

In [ ]:
kcore = versatility.get_multi_Kcore_centrality(supra, nodes=N, layers=L)

# The per-layer indices the minimum is taken over.
layer_graphs = parsing.supra_adjacency_to_network_list(
    supra, nodes=N, layers=L
)
per_layer = np.column_stack([
    gt.topology.kcore_decomposition(g).get_array() for g in layer_graphs
])

table = pd.DataFrame(per_layer, index=NODE_NAMES, columns=LAYER_NAMES)
table["multilayer (min)"] = kcore
print(table.astype(int))

## Local clustering

In [ ]:
clust = pd.DataFrame({
    "edge-coloured": mesoscale.compute_local_clustering_coefficient(
        supra_ec, nodes=N, layers=L),
    "coupled": mesoscale.compute_local_clustering_coefficient(
        supra, nodes=N, layers=L),
}, index=NODE_NAMES)

print(clust.round(3))
print(f"\nedge-coloured: highest is {clust['edge-coloured'].idxmax()}, "
      f"lowest is {clust['edge-coloured'].idxmin()}")
print(f"coupled:       highest is {clust['coupled'].idxmax()}, "
      f"lowest is {clust['coupled'].idxmin()}")

In [ ]:
ax = clust.plot.bar(figsize=(9, 4), width=0.8)
ax.set_ylabel("local clustering coefficient")
ax.set_title("The same measure on the edge-coloured and the coupled matrix")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

Observed-link clustering uses the edge-coloured matrix. Categorical coupling can create
triangles even without observed edges.

In [ ]:
import scipy.sparse as sps

no_edges = [sps.csr_matrix((N, N)) for _ in range(L)]      # no intra-layer edges at all
coupling_only = parsing.build_supra_adjacency_matrix_from_edge_colored_matrices(no_edges, C_cat, N)

print("edges in this network:", coupling_only.nnz, "(all of them coupling)")
print("local clustering:", mesoscale.compute_local_clustering_coefficient(
    coupling_only, nodes=N, layers=L
))

## Layered communities

`get_mod` fits a layered stochastic block model. It needs one graph whose `weight` edge
property stores the layer index.

In [ ]:
# Planted multiplex: 30 nodes in 3 teams of 10, across 3 layers.
# Within a team, edges are dense; across teams, sparse.
rng = np.random.default_rng(7)
gt.seed_rng(7)

N_P = 30
teams = np.repeat([0, 1, 2], 10)          # planted team of each node
TEAM_NAMES = ["team A", "team B", "team C"]
layer_density = [(0.60, 0.03), (0.50, 0.05), (0.55, 0.02)]   # (within, across) per layer

g_multi = gt.Graph(directed=False)
g_multi.add_vertex(N_P)
layer_of_edge = g_multi.new_edge_property("int")

for layer, (p_in, p_out) in enumerate(layer_density):
    for i in range(N_P):
        for j in range(i + 1, N_P):
            p = p_in if teams[i] == teams[j] else p_out
            if rng.random() < p:
                e = g_multi.add_edge(i, j)
                layer_of_edge[e] = layer

g_multi.ep["weight"] = layer_of_edge   # get_mod reads the layer from ep["weight"]
print(f"planted multiplex: {g_multi.num_vertices()} nodes, {g_multi.num_edges()} edges "
      f"across {len(layer_density)} layers")

In [ ]:
agg = gt.spectral.adjacency(g_multi).toarray()   # all layers merged
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(agg, cmap="Greys", vmax=1)
for k in (10, 20):
    ax.axhline(k - 0.5, color="crimson", lw=0.8)
    ax.axvline(k - 0.5, color="crimson", lw=0.8)
ax.set_title("Aggregate adjacency, nodes sorted by team")
ax.set_xticks([5, 15, 25], TEAM_NAMES)
ax.set_yticks([5, 15, 25], TEAM_NAMES)
plt.tight_layout()
plt.show()

`n_iter` sets the number of fits. `return_state=True` returns the fitted state. A
graph-tool seed makes the results repeatable.

In [ ]:
gt.seed_rng(7)
np.random.seed(7)

modules, modularity, state = mesoscale.get_mod(g_multi, n_iter=20, return_state=True)

vals, counts = np.unique(modules, return_counts=True)
print("blocks found over 20 fits:", dict(zip(vals.tolist(), counts.tolist())))
print(f"modularity: best {max(modularity):.3f}, mean {np.mean(modularity):.3f}")

In [ ]:
blocks = state.get_blocks().get_array()[:N_P]

# Match each recovered block to the team it mostly covers, then score the agreement.
recovered = np.zeros(N_P, dtype=int)
for b in np.unique(blocks):
    members = teams[blocks == b]
    recovered[blocks == b] = np.bincount(members).argmax()
accuracy = np.mean(recovered == teams)

contingency = np.zeros((len(np.unique(blocks)), 3), dtype=int)
for row, b in enumerate(np.unique(blocks)):
    for team in range(3):
        contingency[row, team] = np.sum((blocks == b) & (teams == team))

print(f"recovery accuracy against planted teams: {accuracy:.0%}")
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(contingency, cmap="Blues")
ax.set_xticks(range(3), TEAM_NAMES)
ax.set_yticks(range(len(np.unique(blocks))), [f"block {b}" for b in np.unique(blocks)])
ax.set_title("Recovered block vs planted team")
for i in range(contingency.shape[0]):
    for j in range(3):
        ax.text(j, i, contingency[i, j], ha="center", va="center",
                color="w" if contingency[i, j] > contingency.max() * 0.6 else "k")
plt.tight_layout()
plt.show()

## Shortest-path layer similarity

`get_SP_similarity_matrix` compares shortest-path distances between layers.

In [ ]:
sim_office = topology.get_SP_similarity_matrix(supra_ec, nodes=N, layers=L)
print(pd.DataFrame(sim_office.round(3), index=LAYER_NAMES, columns=LAYER_NAMES))

Large values for unreachable pairs can hide smaller differences between connected
layers.

In [ ]:
dist = []
for layer in range(L):
    d = gt.topology.shortest_distance(layer_graphs[layer]).get_2d_array(
        layer_graphs[layer].get_vertices()).astype(float)
    d[d > 1e8] = 1e8                      # the cap get_SP_similarity_matrix applies
    dist.append(d)
    print(f"{LAYER_NAMES[layer]:9s} unreachable pairs: {int((d >= 1e8).sum()):3d}")

print()
for a in range(L):
    for b in range(a + 1, L):
        norm = np.linalg.norm(dist[a] - dist[b])
        print(f"||D[{LAYER_NAMES[a]}] - D[{LAYER_NAMES[b]}]|| = {norm:12.6g}")

Three added meeting edges make every layer connected for the second calculation.

In [ ]:
MEETINGS_PLUS = INTRA["meetings"] + [
    ("Jo", "Ada"),                        # Jo attends one meeting
    ("Cleo", "Dan"), ("Femi", "Gil"),     # two meetings bridge the three triangles
]

rows_plus = []
for layer_name, pairs in {**INTRA, "meetings": MEETINGS_PLUS}.items():
    layer = LAYER_NAMES.index(layer_name)
    for a, b in pairs:
        rows_plus.append((idx[a], layer, idx[b], layer, 1.0))
        rows_plus.append((idx[b], layer, idx[a], layer, 1.0))

edges_plus = pl.DataFrame(
    rows_plus,
    schema=["node.from", "layer.from", "node.to", "layer.to", "weight"],
    orient="row",
)
tensor_plus = parsing.build_tensor_from_dataframe(edges_plus)
supra_plus = parsing.build_supra_adjacency_matrix_from_tensor(tensor_plus)

sim_plus = topology.get_SP_similarity_matrix(supra_plus, nodes=N, layers=L)
print(pd.DataFrame(sim_plus.round(3), index=LAYER_NAMES, columns=LAYER_NAMES))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, mat, title in zip(axes, [sim_office, sim_plus],
                          ["meetings disconnected", "meetings connected"]):
    im = ax.imshow(mat, cmap="magma", vmin=0, vmax=1)
    ax.set_xticks(range(L), LAYER_NAMES, rotation=30)
    ax.set_yticks(range(L), LAYER_NAMES)
    ax.set_title(title)
    for i in range(L):
        for j in range(L):
            ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center",
                    color="w" if mat[i, j] < 0.6 else "k")
fig.colorbar(im, ax=axes, shrink=0.85)
plt.show()

## Inter-layer assortativity

`inter_layer_assortativity` compares layer degrees with Pearson and Spearman
coefficients.

In [ ]:
assort = mesoscale.inter_layer_assortativity(g_list, L)
pearson = assort["Pearson"]

print("Pearson degree correlation between layers:")
print(pd.DataFrame(pearson.round(3), index=LAYER_NAMES, columns=LAYER_NAMES))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(pearson, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(L), LAYER_NAMES, rotation=30)
ax.set_yticks(range(L), LAYER_NAMES)
ax.set_title("Inter-layer degree assortativity (Pearson)")
for i in range(L):
    for j in range(L):
        ax.text(j, i, f"{pearson[i, j]:.2f}", ha="center", va="center",
                color="w" if abs(pearson[i, j]) > 0.6 else "k")
fig.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout()
plt.show()

## Function guide

| Task | Function | Input |
|---|---|---|
| Label connected parts | `topology.get_connected_components` | coupled supra matrix |
| Find LCC, LIC, and LVC | `topology.get_multi_*` | graph list |
| Measure paths | `topology.get_multi_path_statistics` | coupled supra matrix |
| Measure coreness | `versatility.get_multi_Kcore_centrality` | supra matrix |
| Measure local clustering | `mesoscale.compute_local_clustering_coefficient` | edge-coloured matrix |
| Fit layered groups | `mesoscale.get_mod` | layer-labelled graph |
| Compare layer paths | `topology.get_SP_similarity_matrix` | edge-coloured matrix |
| Compare layer degrees | `mesoscale.inter_layer_assortativity` | graph list |